# 03: Interpretation of the Network Results

In this notebook I connect the metrics from Notebook 02 with the country level patterns I observed in the maps and plots. The focus is on hub concentration, transfer roles, network robustness, the relationship between core size and economic scale, and possible fuel price effects.


In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
from src.data_preprocessing import load_data, correct_coordinates
from src.network_construction import build_graph
from src.network_metrics import assortativity, kplus_core

In [2]:
airports, flights = load_data('../data/Airports.csv', '../data/Flight Data.xlsx')
airports = correct_coordinates(airports)

countries = ['USA', 'China', 'United Kingdom', 'Australia']
graphs = {c: build_graph(flights, c) for c in countries}

## Summary metrics

USA: GDP rank 1, assortativity r is -0.198, and core size is about 41. The network is large, with hub and spoke behaviour and some critical bridge airports.

China: GDP rank 2, assortativity r is -0.397, and core size is about 23. The network is strongly hub focused, especially in the east.

United Kingdom: GDP rank 6, assortativity r is -0.119, and core size is about 16. The network is compact and has several important hubs.

Australia: GDP rank 15, assortativity r is -0.230, and core size is about 9. The network is coastal, with hub and spoke behaviour and long distance links.

I used GDP ranking only as context, not as a formal causal model. It helps compare whether larger economies also have larger domestic network cores.


In [3]:
summary = []
for c in countries:
    G = graphs[c]
    r_val, _, _, _, _ = assortativity(G)
    _, _, peak_rank, _ = kplus_core(G)
    summary.append({
        'Country': c,
        'Nodes': len(G.nodes),
        'Edges': len(G.edges),
        'Assortativity (r)': round(r_val, 3),
        'Core size (r*)': peak_rank
    })

pd.DataFrame(summary)

,Country,Nodes,Edges,Assortativity (r),Core size (r*)
0,USA,663,3004,-0.198,41
1,China,110,580,-0.397,23
2,United Kingdom,60,181,-0.119,16
3,Australia,156,299,-0.230,9


## Interpretation notes

### Spatial structure

The USA network is large and spread across the whole country, with many long distance links. China is much denser in the east and sparser in the west. The UK network is compact, so most domestic links are short. Australia is different because many airports are coastal and long distance routes are needed to connect separated population centres.

### Weighted degree and hub dominance

The USA, China and Australia show a clear King Pauper pattern: a small number of airports carry very high weighted degree while most airports remain much smaller. This suggests strong hub dominance. The UK pattern is less extreme, which fits with a shorter distance, more distributed domestic network.

### Degree and betweenness

A strong relationship between degree and betweenness means that airports with many direct connections are also important transfer points. The USA and China show this clearly. In the USA and Australia, I also found airports with moderate degree and high betweenness, which suggests bridge roles for isolated or remote regions. The UK has more scatter, so direct connectivity and transfer importance are less tightly linked.

### Assortativity and resilience

All four networks are disassortative, meaning high degree airports mainly connect to lower degree airports. This is typical for hub and spoke airline networks. It is efficient for moving traffic through hubs, but it also creates dependence on those hubs. China has the most negative assortativity, while the UK has the weakest negative value and appears less dependent on a small number of hubs.

### Core size and economic scale

The core and periphery result was important for my interpretation. Larger economies in this comparison have larger core sizes: USA 41, China 23, UK 16 and Australia 9. A larger core means more airports share important network functions, which can make the system more robust. A smaller core can be more efficient, but it leaves the network more dependent on fewer key airports.


## Fuel price effects on network evolution

I used the Random Geometric Graph idea from the literature to reason about future changes. In that model, the probability of forming a link depends on distance and a distance penalty parameter α. I treated α as a proxy for fuel price and operating cost.

$$Q_{ij} = \frac{d_{ij}^{-\alpha}}{\sum_{k,l} d_{kl}^{-\alpha}}$$

When fuel price is high, long distance links become more expensive. The USA and Australia are most sensitive to this because their domestic networks cover large distances. In that case, hub and spoke structure may weaken and smaller or medium sized fuel efficient aircraft become more suitable.

When fuel price is low, long distance hub links are easier to maintain. Degree and betweenness become more strongly connected, major hubs dominate transfers, and larger aircraft are more useful on the main routes.

The UK is less sensitive to fuel price changes because domestic distances are shorter. China remains strongly hub focused, but western connectivity is more sensitive because distances are longer and airports are more sparse.

Hub growth still has limits. Real airline networks follow truncated power law behaviour, so physical capacity, slots, cost and operational constraints stop hubs from growing indefinitely.
